# Format whole-body RNN decoder data

Builds the five cross-validation pickle files (`all_fold_0_4sec.pkl` through `all_fold_4_4sec.pkl`) that the training notebook (`02_train_rnns.ipynb`) consumes.

What this notebook does:

- Loads each participant `.mat` file from `RAW_DATA_DIR` (defaults to `../../../../Data/` relative to this folder).
- Reads the new public-release data layout (nested `DataMat` struct, v7 or v7.3) transparently.
- Accepts only the 20 arrays listed in `ARRAY_ORDER`. This single check excludes both `BAD_ARRAYS` (e.g. `T17-m1`, `T11-d2`) and the per-participant `*Sorted` arrays.
- Excludes raw cue IDs 47 and 48 (T16's `EYES Up/Down`, T17's `HUM Hi/Low`).
- Uses the fixed 4-second neural window: `goCue` to `goCue + 200` bins at 20 ms per bin.
- Generates five reproducible folds with `RANDOM_SEED = 0`.

Output: `outputs/formatted_data/all_fold_<i>_4sec.pkl` next to this notebook.

In [ ]:
from whole_body_pipeline import (
    ARRAY_ORDER,
    N_FOLDS,
    OUTPUT_DIR,
    RANDOM_SEED,
    RAW_DATA_DIR,
    WINDOW_BINS,
    format_all_folds,
)

formatted_data_dir = OUTPUT_DIR / 'formatted_data'
print(f'Raw data:         {RAW_DATA_DIR}')
print(f'Formatted output: {formatted_data_dir}')
print(f'Window:           {WINDOW_BINS} bins = {WINDOW_BINS * 0.02:.1f} seconds')
print(f'Folds:            {N_FOLDS}; CV random seed: {RANDOM_SEED}')
print(f'Arrays kept:      {len(ARRAY_ORDER)}')

In [ ]:
output_paths = format_all_folds(
    output_dir=formatted_data_dir,
    raw_data_dir=RAW_DATA_DIR,
    n_folds=N_FOLDS,
    random_seed=RANDOM_SEED,
    window_bins=WINDOW_BINS,
)

for path in output_paths:
    print(path)

In [ ]:
import pickle
import numpy as np

for path in output_paths:
    with path.open('rb') as handle:
        data = pickle.load(handle)
    array_list = data['metadata']['array_list']
    train_trials = [len(array_data['sentenceDat']) for array_data in data['train']]
    test_trials  = [len(array_data['sentenceDat']) for array_data in data['test']]
    labels = np.concatenate(
        [np.concatenate(array_data['phonemes']) for array_data in data['train']]
    )
    print(
        f'{path.name}: arrays={len(array_list)}, '
        f'train trials/array={min(train_trials)}-{max(train_trials)}, '
        f'test trials/array={min(test_trials)}-{max(test_trials)}, '
        f'label range={labels.min()}-{labels.max()}'
    )